# Exploration 02 — CRSP (point-in-time universe + realized quotes + join)

Three roles for CRSP: (a) point-in-time S&P 500 constituents, (b) weights (market cap), (c) prices/returns -> realized vol & correlation.
Plus the identifier join `secid (OptionMetrics) <-> permno (CRSP)`.

Read-only — queries filtered at the source.

In [1]:
from dispersion.data.wrds_client import get_connection

db = get_connection()

Loading library list...


Done


## 1) Available S&P 500 tables + DECISION legacy vs modern

The modern CIZ product `idx_const_*_v2` (membership + weights + returns all-in-one) sits on the `crsp_q_mi_hist` schema -> **access denied** (not in the Imperial subscription). So we go with the **legacy approach** (standard in the literature).

In [2]:
tabs = db.list_tables(library='crsp')
cand = sorted(t for t in tabs if '500' in t or 'dsp' in t.lower() or 'msp' in t.lower())
print('CRSP ~ S&P 500 tables:', cand)

# Check that the modern product is indeed denied (try/except so the notebook still runs)
try:
    db.raw_sql('SELECT MIN(effective_date) FROM crsp.idx_const_close_v2')
    print('idx_const_close_v2 : ACCESSIBLE')
except Exception as e:
    print('idx_const_close_v2 : denied ->', str(e).splitlines()[0][:90])

CRSP ~ S&P 500 tables: ['dsp500', 'dsp500_v2', 'dsp500list', 'dsp500list_v2', 'dsp500p', 'msp500', 'msp500_v2', 'msp500list', 'msp500list_v2', 'msp500p']
idx_const_close_v2 : denied -> (psycopg2.errors.InsufficientPrivilege) permission denied for schema crsp_q_mi_hist


## 2) Point-in-time membership: `crsp.dsp500list` (permno, start, ending)

In [3]:
print(db.describe_table(library='crsp', table='dsp500list'))
print()
print(db.raw_sql('SELECT MIN(start) min_start, MAX(ending) max_end, '
                 'COUNT(DISTINCT permno) n_permno FROM crsp.dsp500list').to_string())
# 1925 -> 2024, 1936 permnos. Covers OptionMetrics widely (1996+).

Approximately 2064 rows in crsp.dsp500list.


     name  nullable     type                                         comment
0  permno      True  INTEGER                           CRSP Permanent Number
1   start      True     DATE    Date when the stock included in S&P500 Index
2  ending      True     DATE  Date when the stock excluded from S&P500 Index

    min_start     max_end  n_permno
0  1925-12-31  2024-12-31      1936


## 3) Realized quotes: `crsp.dsf` (adjusted prices, returns, shares)

In [4]:
# Test on Apple (permno 14593). ret = already-adjusted return; shrout in thousands;
# cfacpr = price adjustment factor; market cap = abs(prc)*shrout.
print(db.raw_sql("SELECT permno, date, prc, ret, shrout, cfacpr FROM crsp.dsf "
                 "WHERE permno = 14593 AND date BETWEEN '2024-12-02' AND '2024-12-10' "
                 "ORDER BY date").to_string())

   permno        date        prc       ret      shrout  cfacpr
0   14593  2024-12-02     239.59  0.009523  15115823.0     1.0
1   14593  2024-12-03  242.64999  0.012772  15115823.0     1.0
2   14593  2024-12-04  243.00999  0.001484  15115823.0     1.0
3   14593  2024-12-05  243.03999  0.000123  15115823.0     1.0
4   14593  2024-12-06     242.84 -0.000823  15115823.0     1.0
5   14593  2024-12-09     246.75  0.016101  15115823.0     1.0
6   14593  2024-12-10     247.77  0.004134  15115823.0     1.0


## 4) The bottleneck: `secid <-> permno` join

`wrdsapps_link_crsp_optionm.opcrsphist`: date-ranged link (`sdate`/`edate`) with a quality `score` (1 = best).

In [5]:
print(db.describe_table(library='wrdsapps_link_crsp_optionm', table='opcrsphist'))
print()
# Example: Apple secid 101594 -> permno 14593 (matches dsf), 1996-2025, score 1.0
print(db.raw_sql('SELECT * FROM wrdsapps_link_crsp_optionm.opcrsphist WHERE secid = 101594').to_string())

Approximately 121773 rows in wrdsapps_link_crsp_optionm.opcrsphist.
     name  nullable              type      comment
0   secid      True  DOUBLE PRECISION  Security ID
1   sdate      True              DATE         None
2   edate      True              DATE         None
3  permno      True           INTEGER       PERMNO
4   score      True  DOUBLE PRECISION         None

      secid       sdate       edate  permno  score
0  101594.0  1996-01-02  2025-12-31   14593    1.0


In [6]:
db.close()